# Success-guided acquisition: new method only

Demo-SCORE-inspired **acquisition adaptation**, not a paper reproduction. Reuse the existing seed-11 baseline and controls. Train a state-only success/failure classifier from separate rollout suites at the saved 40k/50k continued-training checkpoints (both trained on the original 50 demonstrations); validate on the later checkpoint. Average its predictions across each expert episode, then acquire the top 20 remaining episodes per round. Ranking is fixed across three rounds; all previously acquired episodes are retained.

No original arm is retrained or reevaluated on the final suite. Saved continued-training checkpoints generate new classifier rollouts. Default cost: **400 classifier rollouts, three 10k-update training stages, and 600 evaluation rollouts**. All simulation uses original DDPM, execute=8. Classifier fits on CPU; policy training/rollouts benefit from GPU. Existing final starts are reused for an **exploratory** comparison only, since their results informed this redesign.

Source: [Demo-SCORE](https://arxiv.org/html/2503.03707v2). The paper filters heterogeneous demonstrations using classifiers validated across checkpoints. Here we rank additional demonstrations from a fixed pool. Scores are not causal estimates of training benefit or calibrated demonstration success probabilities.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Load data and verify simulator compatibility

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## Locate your existing run
Only change `ORIGINAL` if your Drive directory has a different name. Do not point `OUTPUT` at the original directory. This notebook requires the original frozen protocol, baseline `latest.pt`, and saved continued-training round-2/round-3 checkpoints. Combining results also needs original `results.json` and evaluation summaries/checkpoints.

Keep configuration unchanged to resume. To change classifier settings after inspecting results, choose a new output directory. Pin `REVISION` above to the printed commit for repeatability.

In [ ]:
from dataclasses import asdict
from robot_flywheel.demo_score import (ScoreConfig, prepare_success_guided,
    run_success_guided, compare_success_guided)
ORIGINAL = WORK / 'exploratory_ddpm_v1_seed11'
OUTPUT = WORK / 'success_guided_v3_seed11'
SCORE_CONFIG = ScoreConfig(rollouts_per_checkpoint=200, rollout_start=300000,
                           training_round=2, validation_round=3)
assert (ORIGINAL / 'frozen_protocol.json').exists(), f'Original study missing: {ORIGINAL}'
display(asdict(SCORE_CONFIG))
print('Original:', ORIGINAL, '\nNew outputs:', OUTPUT)


## 1. Fit and validate the classifier
Two hidden layers of eight units, dropout and weight decay. Inputs are agent/object positions and sine/cosine of object angle. Coverage, terminal flags, timestep and outcome are not inputs. Each trajectory has equal loss weight; no frames cross training/validation splits. The training checkpoint uses starts 300000–300199 and validation checkpoint 300200–300399.

At least five successes and five failures are required **in each split**. If insufficient, stop and increase the rollout budget in a new experiment directory. Validation loss must beat a constant predictor using the training success rate before acquisition training is allowed. Passing this gate does not establish that rankings improve policy performance. Validation chooses the classifier checkpoint, so its loss is a development metric, not independent test evidence.

In [ ]:
CLASSIFIER_REPORT = prepare_success_guided(
    EPISODES, SOURCE, ORIGINAL, OUTPUT, COMPATIBILITY, SCORE_CONFIG)
display({k:v for k,v in CLASSIFIER_REPORT.items() if k != 'history'})
import matplotlib.pyplot as plt
h = CLASSIFIER_REPORT['history']
plt.plot([r['step'] for r in h], [r['validation_loss'] for r in h], label='Validation BCE')
plt.axhline(CLASSIFIER_REPORT['constant_validation_loss'], color='gray', linestyle='--', label='Constant predictor')
plt.xlabel('Classifier step'); plt.ylabel('Trajectory-weighted BCE'); plt.legend(); plt.show()


## 2. Inspect ranking before training
Plot high- and low-ranked candidate demonstrations. Check whether scoring mainly separates easy initial positions or near-goal states. Classifier accuracy alone cannot prove useful selection. These are development diagnostics; keep all outcomes, including regressions.

In [ ]:
ranking = json.loads((OUTPUT / 'ranking.json').read_text())
base = torch.load(ORIGINAL / 'seed_11/baseline/latest.pt', map_location='cpu', weights_only=False)
initial = set(base['manifest']['episode_ids'])
candidates = [r for r in ranking if r['episode_id'] not in initial]
display(candidates[:20])
by_id = {e.id:e for e in EPISODES}
fig, axs = plt.subplots(1,2,figsize=(12,5))
for ax, rows, title in zip(axs, [candidates[:5],candidates[-5:]], ['Highest scores','Lowest scores']):
    for row in rows:
        ep=by_id[row['episode_id']]
        ax.plot(ep.states[:,2],ep.states[:,3],label=f"{ep.id}: {row['score']:.3f}")
        ax.scatter(*ep.states[0,2:4],s=15)
    ax.set(xlim=(0,512),ylim=(0,512),aspect='equal',title=title);ax.legend()
plt.show()


## 3. Train only the new arm
Branches from the original 20k-update baseline, preserving weights, optimizer, EMA, normalization and RNG state. Three rounds add 20 episodes each and train 10k updates each. Selection records include classifier and parent hashes. Rerun unchanged to resume interrupted training; no original checkpoints are overwritten.

In [ ]:
CHECKPOINTS = run_success_guided(
    EPISODES, SOURCE, ORIGINAL, OUTPUT, COMPATIBILITY, final_evaluation=False)
display(CHECKPOINTS)


## 4. Evaluate only the three new checkpoints
200 paired starts per round (600 rollouts total). Cached rollouts are reused after interruption. The original controls are read later, never reevaluated here. This exploratory suite cannot serve as an untouched confirmation after redesign; freeze the method and run fresh starts and additional training seeds for confirmation.

In [ ]:
RESULTS = run_success_guided(
    EPISODES, SOURCE, ORIGINAL, OUTPUT, COMPATIBILITY, final_evaluation=True)
display(RESULTS)


## 5. Append the new method to the existing comparisons
Creates new `combined_results.csv`, `combined_results.json`, `comparisons.json`, and `combined_curves.png` under OUTPUT. Original reports remain untouched. Confidence intervals are conditional on this one training seed; no training-seed robustness claim is supported.

In [ ]:
COMPARISONS, fig = compare_success_guided(ORIGINAL, OUTPUT)
plt.show()
display(COMPARISONS)
import pandas as pd
display(pd.read_csv(OUTPUT / 'combined_results.csv'))


## 6. Paired example videos and downloadable artifacts

In [ ]:
from IPython.display import Video, FileLink
old_records=json.loads((ORIGINAL/'results.json').read_text())['records']
for name, records in [('random',old_records),('targeted',old_records),('success_guided',RESULTS)]:
    record=max((r for r in records if r['arm']==name),key=lambda r:r['round'])
    directory=Path(record['evaluation']).parent
    start=json.loads((directory/'summary.json').read_text())['seeds'][0]
    video=directory/f'episode_{start}.mp4'
    print(name)
    if video.exists(): display(Video(str(video),embed=True,width=360))
    else: print('No saved video for this start.')
for name in ['classifier_report.json','ranking.json','combined_results.csv','comparisons.json','combined_curves.png']:
    display(FileLink(str(OUTPUT/name)))
print('Artifacts:', OUTPUT)
